In [1]:
# install the kaggle library
!pip install kaggle

Now I upload kaggle.json

In [2]:
# configuring the path of kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [3]:
# Import Twitter sentiment dataset
# API to fetch dataset from kaggle
!kaggle datasets download -d kazanova/sentiment140



Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
  0% 0.00/80.9M [00:00<?, ?B/s]
100% 80.9M/80.9M [00:00<00:00, 1.10GB/s]


In [4]:
# Extracting compressed dataset
!unzip sentiment140.zip
print("Data is extracted")

Archive:  sentiment140.zip
  inflating: training.1600000.processed.noemoticon.csv  
Data is extracted


In [5]:
! pip install nltk scikit-learn

In [6]:
# Download stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Now Import The Dependencies


In [7]:
import numpy as np
import pandas as pd
import re    # remove special characters
from nltk.corpus import stopwords # remove stopwords i,am
from nltk.stem.porter import PorterStemmer # Extract root of the word
from sklearn.feature_extraction.text import  TfidfVectorizer # convert text into numbers
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score



In [8]:
# print stop words in english
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data Processing


In [10]:
# load data from csv into data frame
df = pd.read_csv("/content/training.1600000.processed.noemoticon.csv",encoding='ISO-8859-1')

In [11]:
df.shape

(1599999, 6)

In [12]:
print(df.head())

   0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY _TheSpecialOne_  \
0  0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   scotthamilton   
1  0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY        mattycus   
2  0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY         ElleCTF   
3  0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY          Karoli   
4  0  1467811372  Mon Apr 06 22:20:00 PDT 2009  NO_QUERY        joy_wolf   

  @switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D  
0  is upset that he can't update his Facebook by ...                                                                   
1  @Kenichan I dived many times for the ball. Man...                                                                   
2    my whole body feels itchy and like its on fire                                                                    
3  @nationwideclass no, it's not behaving at all....           

In [13]:
# Now we need to specify the columns names
columns = ['target','ids','date','flag','user','text']
df = pd.read_csv("/content/training.1600000.processed.noemoticon.csv",names=columns,encoding='ISO-8859-1')


In [14]:
df.shape

(1600000, 6)

In [15]:
print(df.head())

   target         ids                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


In [17]:
# Check missing values
print(df.isnull().sum())

target    0
ids       0
date      0
flag      0
user      0
text      0
dtype: int64


In [20]:
# check the distribution of target column
print(df['target'].value_counts())

target
0    800000
4    800000
Name: count, dtype: int64


see, the target column distribution is even(equal) ,iftarget column distribution is uneven(unequal)  we have to do things like upsampling,downsampling

In [22]:
from tkinter import TRUE
# 0 means negative , 4 means positive
# now i need to replace 1 instead of 4
df.replace({'target':{4:1}}, inplace=True)


In [24]:
# 0 means negative , 1 means positive
print(df['target'].value_counts())

target
0    800000
1    800000
Name: count, dtype: int64


**Stemming:**
stemming is a process reduce a word to its root word


In [25]:
port_stem = PorterStemmer()


In [26]:
# stemming function
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content) # remove special characters
  stemmed_content = stemmed_content.lower() # convert lowercase
  stemmed_content = stemmed_content.split() # split mans make a list of line
  final_words=[]
  for word in  stemmed_content:
    if not word in stopwords.words('english'):
      final_words.append(port_stem.stem(word)) # port_stem.stem convert into root word
  stemmed_content = ' '.join(final_words) # ' ' means put space between words
  return stemmed_content





In [27]:
# create new column in original dataset
df['stemmed_content'] = df['text'].apply(stemming)

In [28]:
print(df.head())

   target         ids                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  \
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1    scotthamilton  is upset that he can't update his Facebook by ...   
2         mattycus  @Kenichan I dived many times for the ball. Man...   
3          ElleCTF    my whole body feels itchy and like its on fire    
4           Karoli  @nationwideclass no, it's not behaving at all....   

                                     stemmed_content  
0  switchfoot http twitpic com zl awww bummer sho...  
1  upset updat facebook text might cri result sch...  
2  ke

In [29]:
print(df['stemmed_content'])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemmed_content, Length: 1600000, dtype: object


In [30]:
print(df['target'])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


In [31]:
# seperate the X data and y labels
X = df['stemmed_content'].values
Y = df['target'].values

In [41]:
print(X)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [42]:
print(Y)

[0 0 0 ... 1 1 1]


In [43]:
X_train,X_test,Y_tarin,Y_test = train_test_split(X,Y,test_size = 0.3,stratify = Y,random_state = 14)

In [44]:
print(X.shape,X_train.shape,X_test.shape)

(1600000,) (1120000,) (480000,)


In [45]:
# convert textual data into numerical data
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [46]:
print(X_train_vec)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 8271321 stored elements and shape (1120000, 423004)>
  Coords	Values
  (0, 265947)	0.7257641715024228
  (0, 110357)	0.35103055831147084
  (0, 266648)	0.2801991199295473
  (0, 283177)	0.36515452896737427
  (0, 146552)	0.37174525909587747
  (1, 102924)	0.562811289397428
  (1, 261822)	0.5916176205639845
  (1, 350392)	0.3171378159439574
  (1, 160743)	0.3303020715832254
  (1, 341888)	0.35150560560415045
  (2, 97220)	0.47573342745666264
  (2, 306864)	0.15573180510921875
  (2, 221846)	0.13811859457485068
  (2, 315764)	0.3163076008381814
  (2, 264245)	0.16050513416068832
  (2, 412548)	0.37631015850014826
  (2, 252548)	0.19305685720039262
  (2, 140590)	0.33677119469751815
  (2, 182617)	0.21270798320443152
  (2, 402404)	0.15964301716992196
  (2, 2985)	0.3172098975648063
  (2, 391349)	0.20236467761252075
  (2, 118610)	0.2781483542774462
  (2, 276626)	0.15023440782467296
  (3, 402404)	0.20229182846474336
  :	:
  (1119997, 391349)	0.2810

In [47]:
print(X_test_vec)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3427033 stored elements and shape (480000, 423004)>
  Coords	Values
  (0, 138973)	0.7834815729972668
  (0, 155404)	0.3531486634647843
  (0, 302160)	0.5113146255161933
  (1, 3011)	0.2650953371367151
  (1, 11472)	0.2667235852780111
  (1, 62459)	0.2622367218980233
  (1, 104619)	0.40536161916689356
  (1, 138233)	0.23068412681391864
  (1, 218022)	0.44494339514610237
  (1, 295965)	0.33154875017941987
  (1, 344375)	0.25246709977440845
  (1, 406225)	0.367995894583754
  (1, 416815)	0.2553857294110487
  (2, 57056)	0.48262955729715773
  (2, 57334)	0.27718243002368154
  (2, 60884)	0.3153294056191242
  (2, 202212)	0.16207022071310614
  (2, 209055)	0.22262187721873375
  (2, 286547)	0.3638880201302531
  (2, 354280)	0.23299658967211823
  (2, 369720)	0.5229903327209586
  (2, 406193)	0.23400412408777413
  (3, 5780)	0.22261087798020862
  (3, 28651)	0.3266757127245409
  (3, 51402)	0.3986120532840418
  :	:
  (479995, 154088)	0.39147127931118936


**Training the machine learning model**

Logistic Regression

In [48]:
model = LogisticRegression(max_iter=1000)

In [49]:
model.fit(X_train_vec,Y_tarin)

LogisticRegression(max_iter=1000)

Model Evaluation

In [51]:
# now we check training data accuracy
traindata_predict = model.predict(X_train_vec)
accuracy_traindata = accuracy_score(traindata_predict,Y_tarin)
print(accuracy_traindata)

0.7990946428571428


In [55]:
# now we check test data accuracy ..check whether our model is overfit or not
testdata_predict = model.predict(X_test_vec)
accuracy_testdata = accuracy_score(testdata_predict,Y_test)
print(accuracy_testdata)

0.7756458333333334


Test data accuracy is 77% and training data accuracy is 79%..so model is accurate ..no overfitting exists

**Saved the train model**

In [56]:
import pickle

In [61]:
filename = 'trained_model.sav'
pickle.dump(model,open(filename,'wb'))

Using the saved model for future predictions

In [62]:
# loading the ssaved model
loaded_model = pickle.load(open('/content/trained_model.sav','rb'))


In [71]:
X_new = X_test_vec[900]
print(Y_test[900])
new_predict = loaded_model.predict(X_new)
print(new_predict)

if new_predict[0] == 0:
  print("Negative Tweet")
else:
    print("Positive Tweet")



0
[0]
Negative Tweet


In [72]:
X_new = X_test_vec[200]
print(Y_test[200])
new_predict = loaded_model.predict(X_new)
print(new_predict)

if new_predict[0] == 0:
  print("Negative Tweet")
else:
    print("Positive Tweet")

1
[1]
Positive Tweet
